In [1]:
# Core Python imports
import os
import datetime
import functools
import warnings
warnings.filterwarnings("ignore",message="In a future version of xarray the default value for join",category=FutureWarning)
warnings.filterwarnings("ignore",message="In a future version of xarray the default value for compat",category=FutureWarning)
warnings.filterwarnings("ignore",message="Data requested at a higher resolution than available")
warnings.filterwarnings("ignore",message="Importing `spectral_angle_mapper` from `torchmetrics.functional`",category=FutureWarning)
# Scientific standard imports
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# PyEarthTools imports including NCI cached data
import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
from pyearthtools.data.time import Petdt

my_site = 'site_archive_nci'  # set this to 'site_archive_nci', 'site_archive_jasmin' or 'site_archive_met_office'
import importlib
_ = importlib.import_module(my_site)

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)

# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Detected device {device}, will attempt to use for training")

Detected device cuda:0, will attempt to use for training


In [2]:
# We specify the date, hour, and minute for querying data
date = '20200105T0000'
start_date = '20200101T00'
end_date = '20210101T00'
sat_timestep = '10 minutes'

# pipe_iterator = petpipe.iterators.DateRange(start_date, end_date, interval=sat_timestep)

In [3]:
def _make_iterator(start, end, timestep):
    return petpipe.iterators.DateRange(start, end, interval=timestep)

# Himawari

In [4]:
himawar_vars = [
    'surface_global_irradiance',
    # 'solar_elevation'
]
himawari = petdata.archive.Himawari(himawar_vars)

temporal_window_sat = petpipe.modifications.TemporalWindow(
    prior_indexes=[i for i in range(-12, 0)],
    posterior_indexes=[0],
    merge_method = functools.partial(xr.concat, dim='time'),
    timedelta=petdata.time.TimeDelta((10, "minutes"))
)
sat_pipe = petpipe.Pipeline(
    himawari,
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),  #
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),
    petdata.transform.region.Bounding(-35, -28.5, 145, 151.5),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    temporal_window_sat,
    petpipe.operations.xarray.Merge(),
    # iterator=_make_iterator(start_date, end_date, sat_timestep),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError
)

sample_sat = sat_pipe[date]["surface_global_irradiance"]
like_himawari = petdata.transform.region.like(sample_sat)

# BARRA

In [5]:
barra_convective_vars = [
    "RH24mean"
]
barra_conv = petdata.archive.BARRA_V2(barra_convective_vars, domain_id='AUST-11', frequency='1hr') #, transforms=like_himawari)

temporal_window_bar = petpipe.modifications.TemporalWindow(
    prior_indexes=[i for i in range(-2, 0)],
    posterior_indexes=[0],
    merge_method = functools.partial(xr.concat, dim='time'),
    timedelta=petdata.time.TimeDelta((1, "h"))
)

bar_pipe = petpipe.Pipeline(
    barra_conv,
    petdata.transforms.coordinates.Drop("crs"),
    petpipe.operations.xarray.Sort(order=['time', 'latitude', 'longitude']),
    # Align the data variable's coordinate order to the dataset coordinate order so all arrays are the same shape
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),
    petdata.transform.region.Bounding(-35, -28.5, 145, 151.5),  # cut down on region for example
    petpipe.operations.xarray.normalisation.SingleValueDivision(100),
    temporal_window_bar,
    petpipe.operations.xarray.Merge(),
    # iterator=_make_iterator(start_date, end_date, "1 hour"),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

# Merging

In [6]:
full_pipe = petpipe.Pipeline(
    (sat_pipe, bar_pipe),
    petpipe.operations.xarray.join.InterpLike(sample_sat, method="nearest"),
    petdata.transform.region.Bounding(-34.4, -29.3, 146.1, 151.2), # bound again after interpolation, to remove nans on region boundary
    # petpipe.operations.xarray.conversion.ToNumpy(),
    # petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    # iterator=_make_iterator(start_date, end_date, sat_timestep),
)

In [7]:
plot = False

if plot:
    sample = next(iter(full_pipe))
    fig, ax = plt.subplots(nrows = len(sample.data_vars), ncols=len(sample.time), figsize=(16,2))
    for i, var in enumerate(sample.data_vars):
        for j, t in enumerate(sample.time):
            sample[var].sel(time=t).plot(ax=ax[i,j], add_colorbar=False)
            ax[i,j].set_axis_off()
            ax[i,j].set_title(None)

In [14]:
full_pipe = petpipe.Pipeline(
    (sat_pipe, bar_pipe),
    petpipe.operations.xarray.join.InterpLike(sample_sat, method="nearest"),
    petdata.transform.region.Bounding(-34.4, -29.3, 146.1, 151.2), # bound again after interpolation, to remove nans on region boundary
    # petpipe.operations.xarray.conversion.ToNumpy(),
    # petpipe.operations.numpy.reshape.Rearrange('c t h w -> t c h w'), # channel time height width -> time channel height width
    # iterator=_make_iterator(start_date, end_date, sat_timestep),
)

In [16]:
dates = [
    "20200101T0000",
    "20200101T0300",
    "20200105T0000"
]
for d in dates:
    print(full_pipe[d].time)

/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


<xarray.DataArray 'time' (time: 13)> Size: 104B
array(['2020-01-04T22:00:00.000000000', '2020-01-04T22:10:00.000000000',
       '2020-01-04T22:20:00.000000000', '2020-01-04T22:30:00.000000000',
       '2020-01-04T22:40:00.000000000', '2020-01-04T22:50:00.000000000',
       '2020-01-04T23:00:00.000000000', '2020-01-04T23:10:00.000000000',
       '2020-01-04T23:20:00.000000000', '2020-01-04T23:30:00.000000000',
       '2020-01-04T23:40:00.000000000', '2020-01-04T23:50:00.000000000',
       '2020-01-05T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 104B 2020-01-04T22:00:00 ... 2020-01-05
Attributes:
    long_name:  time
    comment:    Start time of the corresponding satellite observation period


/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


<xarray.DataArray 'time' (time: 13)> Size: 104B
array(['2020-01-04T22:00:00.000000000', '2020-01-04T22:10:00.000000000',
       '2020-01-04T22:20:00.000000000', '2020-01-04T22:30:00.000000000',
       '2020-01-04T22:40:00.000000000', '2020-01-04T22:50:00.000000000',
       '2020-01-04T23:00:00.000000000', '2020-01-04T23:10:00.000000000',
       '2020-01-04T23:20:00.000000000', '2020-01-04T23:30:00.000000000',
       '2020-01-04T23:40:00.000000000', '2020-01-04T23:50:00.000000000',
       '2020-01-05T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 104B 2020-01-04T22:00:00 ... 2020-01-05
Attributes:
    long_name:  time
    comment:    Start time of the corresponding satellite observation period


/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(
/opt/conda/envs/pet/lib/python3.12/site-packages/pyearthtools/data/indexes/_indexes.py:789: IndexWarning: Data requested at a higher resolution than available. minute > hour
  warnings.warn(


<xarray.DataArray 'time' (time: 13)> Size: 104B
array(['2020-01-04T22:00:00.000000000', '2020-01-04T22:10:00.000000000',
       '2020-01-04T22:20:00.000000000', '2020-01-04T22:30:00.000000000',
       '2020-01-04T22:40:00.000000000', '2020-01-04T22:50:00.000000000',
       '2020-01-04T23:00:00.000000000', '2020-01-04T23:10:00.000000000',
       '2020-01-04T23:20:00.000000000', '2020-01-04T23:30:00.000000000',
       '2020-01-04T23:40:00.000000000', '2020-01-04T23:50:00.000000000',
       '2020-01-05T00:00:00.000000000'], dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 104B 2020-01-04T22:00:00 ... 2020-01-05
Attributes:
    long_name:  time
    comment:    Start time of the corresponding satellite observation period
